In [ ]:
# - By default trains nnUNet only; set RUN_DEFORMNET=True to run steps after nnUNet

# 0) Minimal config (EDIT THESE)
KAGGLE_JSON = {"username":"cody11null","key":"913c3a99d13f0c593114f6ae9feae576"}
RUN_DEFORMNET = False  # set True to run steps after nnUNet (merge OOF, soft OOF, DeformNet, archiving)

# 1) Write kaggle.json (instead of Drive copy)
import os, json, subprocess, shutil, time
from pathlib import Path

os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
with open(os.path.expanduser("~/.kaggle/kaggle.json"), "w") as f:
    json.dump(KAGGLE_JSON, f, indent=2)  # fixed: write dict as JSON
os.chmod(os.path.expanduser("~/.kaggle/kaggle.json"), 0o600)
!ls -l ~/.kaggle

# 2) Install dependencies
!pip install -q kaggle nnunetv2 simpleitk scikit-image tqdm pyyaml
!pip install -r requirements.txt
!pip install -e ./src/nnunet/

# 3) Download data
!python3 src/utils/kaggle_helper.py download-competition \
    --competition vesuvius-challenge-surface-detection \
    --out data/

!python3 src/utils/kaggle_helper.py download-dataset \
    --dataset p4rallax/vesuvius-coarse-nnunet-baseline \
    --out nnunet_results/

# 4) Build nnUNet dataset
!NNUNet_raw=./src/nnunet/nnUNet_raw_data_base/nnUNet_raw \
 NNUNet_preprocessed=./src/nnunet/preprocessed \
 NNUNet_results=./src/nnunet/nnUNet_results \
 python3 src/nnUNet_utils/build_nnunet_dataset.py

# 5) Preprocess
!python3 src/nnUNet_utils/nnunet_preprocess.py

print('Done!')

total 4
-rw------- 1 root root 75 Dec 23 21:22 kaggle.json


In [ ]:
# 6) Set canonical nnUNet v2 env (lowercase)
import os
os.environ["nnUNet_raw"] = "./nnunet/nnUNet_raw_data_base/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "./nnunet/preprocessed"
os.environ["nnUNet_results"] = "./nnunet/nnUNet_results"
os.environ["NNUNET_COMPILE"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"
print(os.environ["nnUNet_raw"])
print(os.environ["nnUNet_preprocessed"])
print(os.environ["nnUNet_results"])
if "TORCH_LOGS" in os.environ:
    del os.environ["TORCH_LOGS"]

# 7) Train nnUNet (fold 0; uncomment others if needed)
!nnUNetv2_train Dataset900_VesuviusScroll 3d_fullres 0 -p nnUNetResEncUNetMPlans_30G
# !nnUNetv2_train Dataset900_VesuviusScroll 3d_fullres 1 -p nnUNetResEncUNetMPlans_30G
# !nnUNetv2_train Dataset900_VesuviusScroll 3d_fullres 2 -p nnUNetResEncUNetMPlans_30G
# !nnUNetv2_train Dataset900_VesuviusScroll 3d_fullres 3 -p nnUNetResEncUNetMPlans_30G
# !nnUNetv2_train Dataset900_VesuviusScroll 3d_fullres 4 -p nnUNetResEncUNetMPlans_30G

print('Done with nnUNet!')